# ChainSight on IBM LinuxONE

Real time delivery risk, live supplier scoring and grounded GenAI recommendations.

Run the cells top to bottom (Shift + Enter). Each step calls a script from this folder, so the notebook and the terminal give the same results.

| Step | What happens |
| --- | --- |
| 1 | Check the platform (s390x) and packages |
| 2 | Load data (real DataCo CSV if present, otherwise sample data) |
| 3 | Train delivery and breach models |
| 4 | Look inside the models: calibration, delivery bands, suppliers |
| 5 | Score one shipment end to end: drivers, options, GenAI review, validator |
| 6 | Replay a live event stream |
| 7 | Evaluation report |

## 1. Platform check
Screenshot this output for the judges: it shows the code runs on s390x (LinuxONE).

In [ ]:
%run check_env.py

## 2. Data
Put `DataCoSupplyChainDataset.csv` in the `data/` folder to use the real dataset. Without it, the next cell generates sample data with the same columns.

In [ ]:
from config import RAW_CSV
if RAW_CSV.exists():
    print("Using", RAW_CSV)
else:
    print("Real DataCo CSV not found, generating sample data")
    %run make_sample_data.py --orders 8000
%run prepare_data.py

## 3. Train the models
Three quantile models give early (P10), likely (P50) and late (P90) delivery days. A gradient boosted classifier gives the SLA breach probability, calibrated with isotonic regression on the validation period. The split is by time, so the model never sees the future.

In [ ]:
%run train.py
import json, pandas as pd
from config import ART_DIR
pd.Series(json.loads((ART_DIR / "metrics.json").read_text())).to_frame("value")

## 4. Inside the models

In [ ]:
%matplotlib inline
import numpy as np, pandas as pd, json
from config import ORDERS_PKL, ART_DIR, PRIOR_K
from features import build_matrix
from scoring import Scorer
from books import SupplierBook

orders = pd.read_pickle(ORDERS_PKL)
split = json.loads((ART_DIR / "split.json").read_text())
hist = orders[orders.order_ts <= pd.Timestamp(split["val_end"])]
test = orders[orders.order_id.isin(split["test_order_ids"]) & (orders.canceled == 0)].copy()

scorer = Scorer()
book = SupplierBook(hist)
q, p = scorer.predict(build_matrix(test, scorer.enc))
test["eta_p10"], test["eta_p50"], test["eta_p90"], test["breach_prob"] = q[:, 0], q[:, 1], q[:, 2], p
print(f"{len(test)} test shipments scored with backend: {scorer.backend}")

### Calibration: when the model says 80% risk, about 80% should be late

In [ ]:
bins = pd.cut(test.breach_prob, [0, .2, .4, .6, .8, 1.0], include_lowest=True)
calib = test.groupby(bins, observed=True).agg(predicted=("breach_prob", "mean"), actual_late_rate=("late", "mean"), shipments=("late", "size")).round(3)
display(calib)
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(4.5, 4))
    ax.plot([0, 1], [0, 1], "--", color="grey", label="perfect calibration")
    ax.plot(calib.predicted, calib.actual_late_rate, "o-", label="ChainSight")
    ax.set_xlabel("Predicted breach probability"); ax.set_ylabel("Actual late rate"); ax.legend(); plt.show()
except ImportError:
    print("matplotlib not installed: the table above is enough")

### Delivery bands: P10 to P90 range against what really happened

In [ ]:
sample = test.sample(40, random_state=1).sort_values("eta_p50").reset_index(drop=True)
inside = ((test.real_days >= test.eta_p10) & (test.real_days <= test.eta_p90)).mean()
print(f"Actual delivery inside the P10 to P90 band: {inside:.1%} of test shipments")
try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.vlines(sample.index, sample.eta_p10, sample.eta_p90, color="#9db7e0", lw=6, label="P10 to P90 band")
    ax.plot(sample.index, sample.eta_p50, "_", color="#2f6fd6", ms=12, label="P50")
    ax.plot(sample.index, sample.real_days, "o", color="black", ms=4, label="actual")
    ax.set_xlabel("shipments"); ax.set_ylabel("delivery days"); ax.legend(loc="lower right", fontsize=8); plt.show()
except ImportError:
    display(sample[["ship_mode", "eta_p10", "eta_p50", "eta_p90", "real_days"]])

### Supplier scores with Bayesian shrinkage
`on_time_rate = (on_time + K x prior) / (orders + K)` with K = 20 by default (`PRIOR_K` in `config.py`). A newcomer with a few perfect orders stays near the average and cannot outrank a proven supplier.

In [ ]:
display(book.table().head(10))
prior = book.prior_on_time
newcomer = (2 + PRIOR_K * prior) / (2 + PRIOR_K)
proven = (480 + PRIOR_K * prior) / (500 + PRIOR_K)
print(f"Prior on time rate {prior:.2f}")
print(f"Newcomer with 2 of 2 on time      -> {newcomer:.2f}")
print(f"Proven supplier with 480 of 500   -> {proven:.2f}")

## 5. One shipment end to end
Pick the riskiest high criticality test shipment, explain the score, rescore the fix options counterfactually, then let the GenAI reviewer decide. Its answer must cite evidence ids from the packet.

In [ ]:
from decision import options, priority, needs_approval
row = test.sort_values(["criticality", "breach_prob"], ascending=False).iloc[0].to_dict()
row["sup_late_rate"] = book.late_rate(row["supplier_id"]); row["sup_n_orders"] = book.n_orders(row["supplier_id"])
base = scorer.score(row)
print(f"Shipment {row['order_id']} | {row['department']} | {row['supplier_name']} | {row['ship_mode']} | criticality {row['criticality']}")
print(f"Delivery days P10/P50/P90: {base['eta_p10']:.1f} / {base['eta_p50']:.1f} / {base['eta_p90']:.1f} (promised {row['scheduled_days']:.0f})")
print(f"Calibrated breach probability: {base['breach_prob']:.2f}, scored in {base['score_ms']} ms")
display(pd.DataFrame(base["drivers"]))
opts = options(scorer, book, row, base)
display(pd.DataFrame(opts))

In [ ]:
from reasoner import evidence_packet, review, validate
from config import LLM_BACKEND
sup = {"name": row["supplier_name"], "score": book.score(row["supplier_id"]), "n_orders": book.n_orders(row["supplier_id"])}
packet = evidence_packet({"shipment_id": int(row["order_id"]), "criticality": int(row["criticality"]), "ship_mode": row["ship_mode"],
                          "scheduled_days": float(row["scheduled_days"])}, base, base["drivers"], [], opts, sup, 3.0, [])
answer = review(packet, opts)
print("LLM backend:", LLM_BACKEND, "| answer source:", answer.get("source"))
print(json.dumps(answer, indent=2))
print("Needs one click approval:", needs_approval(answer["recommended_action"], row["criticality"]))

### The validator in action
A made up citation or an action that is not one of the options is rejected, and the system falls back to a rule based alert.

In [ ]:
bad = dict(answer, citations=["model_breach", "invented_fact_42"], recommended_action="teleport")
print(validate(bad, packet))

## 6. Live replay
For the demo, run the replay and dashboard in two JupyterLab terminals (File > New > Terminal):

```
python replay.py --n 1500 --speed 20000 --reset
streamlit run app.py --server.port 8501 --server.address 0.0.0.0
```

The next cell runs a short replay with no waiting, so the evaluation below has data.

In [ ]:
%run replay.py --n 400 --speed 0 --reset

## 7. Evaluation report
Only use numbers from this report in the slides. Results from the mock reviewer are not GenAI results.

In [ ]:
%run evaluate.py